In [5]:
# ============================================================
# P-CULTA
# INTER-ANNOTATOR AGREEMENT
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score


# ============================================================
# 1. LOAD ANNOTATION FILE
# ============================================================

INPUT_FILE = "gold_curation_annotation.csv"

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("P-CULTA INTER-ANNOTATOR AGREEMENT")
print("=" * 70)

print(f"Total annotations: {len(df)}")


# ============================================================
# 2. VERIFY DATASET SIZE
# ============================================================

if len(df) != 918:
    raise ValueError(
        f"Expected 918 rows "
        f"(306 scenarios × 3 candidates), "
        f"but found {len(df)}."
    )

print("✓ 918 candidate annotations confirmed.")
print("✓ 306 scenarios × 3 candidates.")


# ============================================================
# 3. DIMENSIONS
# ============================================================

CAS_DIMS = [
    "Relational_Alignment",
    "Indirectness",
    "Facesaving",
    "Cultural_Appropriateness"
]

FTV_DIMS = [
    "Directness_Threat",
    "Hierarchy_Violation"
]

ALL_DIMS = CAS_DIMS + FTV_DIMS


# ============================================================
# 4. VERIFY COLUMNS
# ============================================================

required_columns = []

for dim in ALL_DIMS:

    required_columns.append(f"A_{dim}")
    required_columns.append(f"B_{dim}")


missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "The following required annotation columns "
        "are missing:\n"
        + "\n".join(missing_columns)
    )

print("✓ All six annotation dimensions found.")


# ============================================================
# 5. CONVERT SCORES TO NUMERIC
# ============================================================

for dim in ALL_DIMS:

    df[f"A_{dim}"] = pd.to_numeric(
        df[f"A_{dim}"],
        errors="coerce"
    )

    df[f"B_{dim}"] = pd.to_numeric(
        df[f"B_{dim}"],
        errors="coerce"
    )


# ============================================================
# 6. CHECK VALID SCORE RANGE
# ============================================================

for dim in ALL_DIMS:

    for annotator in ["A", "B"]:

        col = f"{annotator}_{dim}"

        invalid = df[
            df[col].notna() &
            (
                (df[col] < 0) |
                (df[col] > 4)
            )
        ]

        if len(invalid) > 0:

            raise ValueError(
                f"Invalid scores found in {col}. "
                f"Scores must be between 0 and 4."
            )

print("✓ All annotation scores are within 0–4.")


# ============================================================
# 7. COMPUTE COHEN'S KAPPA
# ============================================================
#
# This follows the same principle as your notebook:
#
#     cohen_kappa_score(
#         annotator_1_scores,
#         annotator_2_scores,
#         weights='quadratic'
#     )
#
# Unlike CAS/FTV, we DO NOT average the annotators here.
#
# Judge 1 ratings are compared directly against Judge 2.
# ============================================================


def compute_kappas(df):

    rows = []

    for dim in ALL_DIMS:

        col_a = f"A_{dim}"
        col_b = f"B_{dim}"

        # ----------------------------------------------------
        # Keep only pairs where BOTH annotators provided
        # a rating.
        # ----------------------------------------------------

        mask = (
            df[col_a].notna() &
            df[col_b].notna()
        )

        r1 = df.loc[
            mask,
            col_a
        ].astype(int).tolist()

        r2 = df.loc[
            mask,
            col_b
        ].astype(int).tolist()

        # ----------------------------------------------------
        # Minimum observations
        # ----------------------------------------------------

        if len(r1) < 5:
            continue

        # ----------------------------------------------------
        # Quadratic weighted Cohen's kappa
        # ----------------------------------------------------

        kappa = cohen_kappa_score(
            r1,
            r2,
            weights="quadratic"
        )

        # ----------------------------------------------------
        # Same strength interpretation as your notebook
        # ----------------------------------------------------

        if kappa > 0.80:

            strength = "Almost Perfect"

        elif kappa > 0.61:

            strength = "Substantial"

        elif kappa > 0.41:

            strength = "Moderate"

        else:

            strength = "Fair"

        # ----------------------------------------------------
        # CAS vs FTV classification
        # ----------------------------------------------------

        metric = (
            "CAS"
            if dim in CAS_DIMS
            else "FTV"
        )

        rows.append({

            "Dimension": dim,

            "Metric": metric,

            "κ": round(
                kappa,
                4
            ),

            "Strength": strength,

            "N": len(r1)

        })

    return pd.DataFrame(rows)


# ============================================================
# 8. RUN AGREEMENT ANALYSIS
# ============================================================

kappas = compute_kappas(df)


# ============================================================
# 9. PRINT RESULTS
# ============================================================

print()
print("=" * 90)
print(
    "INTER-ANNOTATOR AGREEMENT "
    "(QUADRATIC WEIGHTED COHEN'S κ)"
)
print("=" * 90)

print(
    kappas.to_string(
        index=False
    )
)


# ============================================================
# 10. MINIMUM KAPPA CHECK
# ============================================================

min_k = kappas["κ"].min()
max_k = kappas["κ"].max()
mean_k = kappas["κ"].mean()

print()
print("=" * 90)
print("AGREEMENT SUMMARY")
print("=" * 90)

print(
    f"Mean κ      = {mean_k:.4f}"
)

print(
    f"Minimum κ   = {min_k:.4f}"
)

print(
    f"Maximum κ   = {max_k:.4f}"
)


if min_k < 0.61:

    print(
        f"⚠ Minimum κ = {min_k:.4f} "
        "— calibration session recommended"
    )

else:

    print(
        f"✓ All κ ≥ 0.61 — acceptable for publication"
    )


# ============================================================
# 11. SAVE RESULTS
# ============================================================

OUTPUT_FILE = (
    "P_CULTA_inter_annotator_agreement.csv"
)

kappas.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print()
print(f"✓ Saved: {OUTPUT_FILE}")



# ============================================================
# 12. FINAL VALIDATION
# ============================================================

assert len(kappas) == 6

assert set(kappas["Dimension"]) == set(
    ALL_DIMS
)

assert (
    kappas["N"] == 918
).all()

print()
print("=" * 90)
print("FINAL VALIDATION")
print("=" * 90)

print("✓ Six dimensions evaluated.")
print("✓ 918 candidate-level annotation pairs per dimension.")
print("✓ Judge 1 compared directly against Judge 2.")
print("✓ Quadratic weighted Cohen's κ used.")
print("✓ No averaging performed before κ.")
print("✓ Agreement analysis completed successfully.")

P-CULTA INTER-ANNOTATOR AGREEMENT
Total annotations: 918
✓ 918 candidate annotations confirmed.
✓ 306 scenarios × 3 candidates.
✓ All six annotation dimensions found.
✓ All annotation scores are within 0–4.

INTER-ANNOTATOR AGREEMENT (QUADRATIC WEIGHTED COHEN'S κ)
               Dimension Metric      κ       Strength   N
    Relational_Alignment    CAS 0.9800 Almost Perfect 918
            Indirectness    CAS 0.9808 Almost Perfect 918
              Facesaving    CAS 0.9938 Almost Perfect 918
Cultural_Appropriateness    CAS 0.9914 Almost Perfect 918
       Directness_Threat    FTV 0.9955 Almost Perfect 918
     Hierarchy_Violation    FTV 0.9813 Almost Perfect 918

AGREEMENT SUMMARY
Mean κ      = 0.9871
Minimum κ   = 0.9800
Maximum κ   = 0.9955
✓ All κ ≥ 0.61 — acceptable for publication

✓ Saved: P_CULTA_inter_annotator_agreement.csv

FINAL VALIDATION
✓ Six dimensions evaluated.
✓ 918 candidate-level annotation pairs per dimension.
✓ Judge 1 compared directly against Judge 2.
✓ Quadrati

In [6]:
import pandas as pd
import numpy as np


# ============================================================
# 1. LOAD ANNOTATION FILE
# ============================================================

INPUT_FILE = "gold_curation_annotation.csv"

df = pd.read_csv(INPUT_FILE)


# ============================================================
# 2. BASIC VALIDATION
# ============================================================

expected_rows = 306 * 3

if len(df) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} rows (306 scenarios × 3 candidates), "
        f"but found {len(df)} rows."
    )

print(f"Loaded {len(df)} rows.")
print(f"Expected: {expected_rows} rows.")
print("Structure: 306 scenarios × 3 candidates.")


# ============================================================
# 3. SCORE COLUMNS
# ============================================================

dimensions = [
    "Relational_Alignment",
    "Indirectness",
    "Facesaving",
    "Cultural_Appropriateness",
    "Directness_Threat",
    "Hierarchy_Violation"
]

required_columns = (
    ["Candidates"]
    + [f"A_{d}" for d in dimensions]
    + [f"B_{d}" for d in dimensions]
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# 4. CONVERT ANNOTATION SCORES TO NUMERIC
# ============================================================

for dimension in dimensions:
    df[f"A_{dimension}"] = pd.to_numeric(
        df[f"A_{dimension}"],
        errors="coerce"
    )

    df[f"B_{dimension}"] = pd.to_numeric(
        df[f"B_{dimension}"],
        errors="coerce"
    )


# Check for missing scores
score_columns = (
    [f"A_{d}" for d in dimensions]
    + [f"B_{d}" for d in dimensions]
)

missing_scores = df[score_columns].isna().sum()

if missing_scores.sum() > 0:
    print("\nWARNING: Missing annotation scores found:")
    print(missing_scores[missing_scores > 0])
else:
    print("No missing annotation scores found.")


# ============================================================
# 5. CREATE SCENARIO ID
# ============================================================
#
# Every 3 consecutive rows correspond to one scenario:
#
# Scenario 1 -> Candidate A, B, C
# Scenario 2 -> Candidate A, B, C
# ...
# Scenario 306 -> Candidate A, B, C
#

df["Scenario_ID"] = np.arange(len(df)) // 3 + 1


# Candidate labels based on row position
candidate_labels = ["A", "B", "C"]

df["Candidate"] = (
    np.arange(len(df)) % 3
)

df["Candidate"] = df["Candidate"].map({
    0: "A",
    1: "B",
    2: "C"
})


# ============================================================
# 6. FORWARD-FILL SCENARIO-LEVEL INFORMATION
# ============================================================
#
# In the annotation file, fields such as Topic are populated
# for the first candidate and blank for the other two candidates.
#
# We fill them downward within each scenario.
#

scenario_columns = [
    "Language",
    "Topic",
    "User Role",
    "Model Role",
    "Power Distance",
    "Register",
    "Pragmatic Genre",
    "Sensitivity",
    "User Utterance",
    "Context"
]

for col in scenario_columns:
    if col in df.columns:
        df[col] = df.groupby("Scenario_ID")[col].transform(
            lambda x: x.ffill()
        )


# ============================================================
# 7. VERIFY THAT EACH SCENARIO HAS EXACTLY 3 CANDIDATES
# ============================================================

candidate_counts = df.groupby("Scenario_ID").size()

if not (candidate_counts == 3).all():
    bad_scenarios = candidate_counts[
        candidate_counts != 3
    ]

    raise ValueError(
        "Some scenarios do not contain exactly 3 candidates:\n"
        f"{bad_scenarios}"
    )

print("All 306 scenarios contain exactly 3 candidates.")


# ============================================================
# 8. VERIFY ANNOTATION RANGE
# ============================================================
#
# All six dimensions should be in the 0–4 range.
#

for col in score_columns:

    invalid = df[
        (df[col] < 0) |
        (df[col] > 4)
    ]

    if len(invalid) > 0:
        raise ValueError(
            f"Invalid scores found in {col}. "
            "All annotation scores must be between 0 and 4."
        )

print("All annotation scores are within the valid 0–4 range.")


# ============================================================
# 9. AVERAGE THE TWO ANNOTATORS
# ============================================================
#
# For every candidate C and every dimension d:
#
# C_d = (C_d,A1 + C_d,A2) / 2
#
# Here:
# A = Annotator 1
# B = Annotator 2
#

for dimension in dimensions:

    df[f"Mean_{dimension}"] = (
        df[f"A_{dimension}"]
        + df[f"B_{dimension}"]
    ) / 2


# ============================================================
# 10. CALCULATE CAS
# ============================================================
#
# Your exact formula:
#
# CAS_C =
# (C_RA + C_I + C_FS + C_CA) / 16
#
# where every dimension is first averaged across
# the two annotators.
#
# Range: 0–1
# Higher CAS = stronger cultural alignment
#

df["CAS"] = (
    df["Mean_Relational_Alignment"]
    + df["Mean_Indirectness"]
    + df["Mean_Facesaving"]
    + df["Mean_Cultural_Appropriateness"]
) / 16


# ============================================================
# 11. CALCULATE FTV
# ============================================================
#
# Your exact formula:
#
# FTV_C =
# (C_DT + C_HV) / 8
#
# where every dimension is first averaged across
# the two annotators.
#
# Range: 0–1
# Lower FTV = fewer face-threat / hierarchy violations
#

df["FTV"] = (
    df["Mean_Directness_Threat"]
    + df["Mean_Hierarchy_Violation"]
) / 8


# ============================================================
# 12. ROUND SCORES
# ============================================================

df["CAS"] = df["CAS"].round(4)
df["FTV"] = df["FTV"].round(4)


# ============================================================
# 13. SELECT GOLD RESPONSE
# ============================================================
#
# For each scenario:
#
#   1. Higher CAS is preferred.
#   2. If CAS is tied, lower FTV is preferred.
#   3. If both CAS and FTV are exactly tied,
#      mark the scenario as TIE rather than arbitrarily
#      selecting a candidate.
#

def select_gold(group):

    group = group.copy()

    # Sort by:
    #   CAS descending
    #   FTV ascending
    ranked = group.sort_values(
        by=["CAS", "FTV"],
        ascending=[False, True]
    )

    best_cas = ranked.iloc[0]["CAS"]
    best_ftv = ranked.iloc[0]["FTV"]

    # Find candidates with exactly the same CAS and FTV
    tied = ranked[
        (ranked["CAS"] == best_cas) &
        (ranked["FTV"] == best_ftv)
    ]

    if len(tied) > 1:
        return "TIE"

    return ranked.iloc[0]["Candidate"]


gold_candidates = (
    df.groupby("Scenario_ID", group_keys=False)
      .apply(select_gold)
)

gold_candidates.name = "Gold_Candidate"


# ============================================================
# 14. ADD GOLD CANDIDATE TO EVERY ROW OF THE SCENARIO
# ============================================================

df = df.merge(
    gold_candidates,
    on="Scenario_ID",
    how="left"
)


# ============================================================
# 15. IDENTIFY WHETHER EACH ROW IS THE GOLD RESPONSE
# ============================================================

df["Is_Gold"] = (
    df["Candidate"] == df["Gold_Candidate"]
)

# A tie means no candidate is automatically selected
df.loc[
    df["Gold_Candidate"] == "TIE",
    "Is_Gold"
] = False


# ============================================================
# 16. EXTRACT GOLD RESPONSE
# ============================================================

gold_response_lookup = (
    df[df["Is_Gold"]]
    .set_index("Scenario_ID")["Candidates"]
    .to_dict()
)

df["Gold_Response"] = df["Scenario_ID"].map(
    gold_response_lookup
)


# ============================================================
# 17. CREATE A CLEAN CANDIDATE-LEVEL OUTPUT
# ============================================================
#
# This file contains:
#
# Scenario
# Candidate
# Response
# Averaged annotation dimensions
# CAS
# FTV
# Gold Candidate
# Is Gold
#

candidate_output_columns = [
    "Scenario_ID",
    "Candidate",
    "Candidates",

    "Mean_Relational_Alignment",
    "Mean_Indirectness",
    "Mean_Facesaving",
    "Mean_Cultural_Appropriateness",
    "Mean_Directness_Threat",
    "Mean_Hierarchy_Violation",

    "CAS",
    "FTV",

    "Gold_Candidate",
    "Is_Gold"
]

candidate_scores = df[candidate_output_columns].copy()


# ============================================================
# 18. SAVE CANDIDATE-LEVEL SCORES
# ============================================================

candidate_scores.to_csv(
    "candidate_CAS_FTV_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nSaved candidate-level scores to:"
    "\n  candidate_CAS_FTV_scores.csv"
)


# ============================================================
# 19. CREATE ONE ROW PER SCENARIO
# ============================================================
#
# This gives a compact table containing:
#
# Candidate A -> CAS, FTV
# Candidate B -> CAS, FTV
# Candidate C -> CAS, FTV
# Gold Candidate
# Gold Response
#

scenario_output = (
    df.pivot(
        index="Scenario_ID",
        columns="Candidate",
        values=["CAS", "FTV"]
    )
)

# Flatten multi-level column names
scenario_output.columns = [
    f"{metric}_{candidate}"
    for metric, candidate
    in scenario_output.columns
]

scenario_output = scenario_output.reset_index()


# Add scenario metadata from first row of each scenario
metadata_columns = [
    "Language",
    "Topic",
    "User Role",
    "Model Role",
    "Power Distance",
    "Register",
    "Pragmatic Genre",
    "Sensitivity",
    "User Utterance",
    "Context"
]

metadata = (
    df.groupby("Scenario_ID")[metadata_columns]
      .first()
      .reset_index()
)

# Gold candidate information
gold_info = (
    df.groupby("Scenario_ID")
      .first()[[
          "Gold_Candidate",
          "Gold_Response"
      ]]
      .reset_index()
)

scenario_output = (
    metadata
    .merge(scenario_output, on="Scenario_ID")
    .merge(gold_info, on="Scenario_ID")
)


# ============================================================
# 20. REORDER COLUMNS
# ============================================================

desired_order = [
    "Scenario_ID",

    "Language",
    "Topic",
    "User Role",
    "Model Role",
    "Power Distance",
    "Register",
    "Pragmatic Genre",
    "Sensitivity",
    "User Utterance",
    "Context",

    "CAS_A",
    "FTV_A",

    "CAS_B",
    "FTV_B",

    "CAS_C",
    "FTV_C",

    "Gold_Candidate",
    "Gold_Response"
]

# Keep only columns that exist
desired_order = [
    col for col in desired_order
    if col in scenario_output.columns
]

scenario_output = scenario_output[desired_order]


# ============================================================
# 21. SAVE SCENARIO-LEVEL OUTPUT
# ============================================================

scenario_output.to_csv(
    "scenario_CAS_FTV_gold_responses.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved scenario-level results to:"
    "\n  scenario_CAS_FTV_gold_responses.csv"
)


# ============================================================
# 22. PRINT EXAMPLE RESULTS
# ============================================================

print("\n" + "=" * 80)
print("FIRST 5 SCENARIOS")
print("=" * 80)

print(
    scenario_output[
        [
            "Scenario_ID",
            "Topic",
            "CAS_A",
            "FTV_A",
            "CAS_B",
            "FTV_B",
            "CAS_C",
            "FTV_C",
            "Gold_Candidate"
        ]
    ].head(5).to_string(index=False)
)


# ============================================================
# 23. GOLD CANDIDATE COUNTS
# ============================================================

print("\n" + "=" * 80)
print("GOLD CANDIDATE COUNTS")
print("=" * 80)

print(
    scenario_output["Gold_Candidate"]
    .value_counts(dropna=False)
)


# ============================================================
# 24. CHECK FINAL NUMBER OF SCENARIOS
# ============================================================

if len(scenario_output) != 306:
    raise ValueError(
        f"Expected 306 scenarios in final output, "
        f"but found {len(scenario_output)}."
    )

print("\nFinal validation passed: 306 scenarios.")


# ============================================================
# 25. FINAL SCORE RANGES
# ============================================================

print("\n" + "=" * 80)
print("CAS / FTV RANGES")
print("=" * 80)

print(
    f"CAS range: {df['CAS'].min():.4f} – {df['CAS'].max():.4f}"
)

print(
    f"FTV range: {df['FTV'].min():.4f} – {df['FTV'].max():.4f}"
)

Loaded 918 rows.
Expected: 918 rows.
Structure: 306 scenarios × 3 candidates.
No missing annotation scores found.
All 306 scenarios contain exactly 3 candidates.
All annotation scores are within the valid 0–4 range.

Saved candidate-level scores to:
  candidate_CAS_FTV_scores.csv
Saved scenario-level results to:
  scenario_CAS_FTV_gold_responses.csv

FIRST 5 SCENARIOS
 Scenario_ID           Topic  CAS_A  FTV_A  CAS_B  FTV_B  CAS_C  FTV_C Gold_Candidate
           1 Refusing Offers 1.0000    0.0 0.3438 0.2500  0.500 0.1250              A
           2 Refusing Offers 0.9375    0.0 0.3750 0.1875  0.375 0.1875              A
           3 Refusing Offers 1.0000    0.0 0.1875 0.3125  0.250 0.2500              A
           4 Refusing Offers 1.0000    0.0 0.1875 0.2500  0.375 0.1875              A
           5 Refusing Offers 1.0000    0.0 0.2500 0.3750  0.250 0.2500              A

GOLD CANDIDATE COUNTS
Gold_Candidate
A    305
B      1
Name: count, dtype: int64

Final validation passed: 306 s

/tmp/ipykernel_9558/3269756664.py:311: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_gold)
